In [112]:
import math

def coef(n):
    c_n = (-1)**n/(2**n*math.factorial(n))*math.sqrt(math.factorial(2*n+1)/(4*math.pi))
    return 1/((2*n+1)*c_n**2)

coef(2)

1.3404128655316454

In [113]:
import os
import math
import pandas as pd
import numpy as np
print(os.getcwd())
work_path = '/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT'
A=20
Nuclei='Ne'
para_path = os.path.join(work_path,f'examples/{A}{Nuclei}/{A}{Nuclei}_para.dat')
b23_path = os.path.join(work_path,f'examples/{A}{Nuclei}/{A}{Nuclei}_b23.dat')
output_dir = os.path.join(work_path,f'examples/{A}{Nuclei}/output')

# A=150
# Nuclei='Nd'
# para_path = os.path.join(work_path,f'examples/{A}{Nuclei}_term/{A}{Nuclei}_para.dat')
# b23_path = os.path.join(work_path,f'examples/{A}{Nuclei}_term/{A}{Nuclei}_b23.dat')
# output_dir = os.path.join(work_path,f'examples/{A}{Nuclei}_term/output')

/home/yli/MR_CDFT/MRCDFT_2BCorr


In [114]:
# extract eMax, nbeta, nphi from parameter file
with open(para_path, "r", encoding="utf-8") as f:
    for line in f:
        line_nocomment = line.split("!")[0]

        if "n0f" in line_nocomment:
            eMax = int(line_nocomment.split("=")[1].split()[0])
        if "nphi" in line_nocomment:
            nphi = int(line_nocomment.split("=")[1])
        if "nbeta" in line_nocomment:
            nbeta = int(line_nocomment.split("=")[1])
        if "AMP" in line_nocomment:
            AMP = int(line_nocomment.split("=")[1])
        if "PNP" in line_nocomment:
            PNP = int(line_nocomment.split("=")[1])
        if "Kernels" in line_nocomment:
            Kernels = int(line_nocomment.split("=")[1])
    if(AMP == 0): nbeta = 1
    if(PNP == 0): nphi = 1

# eMax = 6; nphi = 7; nbeta = 12; AMP = 1; Kernels = 2

print('eMax=',eMax,' nphi=',nphi,' nbeta=',nbeta,"AMP=",AMP,"PNP=",PNP," Kernels=",Kernels)




eMax= 6  nphi= 1  nbeta= 1 AMP= 0 PNP= 0  Kernels= 2


In [115]:
oneB_ME_path = os.path.join(output_dir,f'mScheme_1B_A{A}_eMax{eMax:02d}.me')
oneB_ME = pd.read_csv(oneB_ME_path,sep=r'\s+')
oneB_ME = oneB_ME.drop(columns=["n1", "n2","l1", "l2","2j1", "2j2","2j_m1", "2j_m2"])
print(oneB_ME_path)
oneB_ME

/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT/examples/20Ne/output/mScheme_1B_A20_eMax06.me


,ifg,m1,m2,r^2,r^4,r^2Y20,r^4Y20,r^4Y40,f2,Eps1B
0,1,1,1,4.11810,28.26461,-0.00000,-0.00000,-0.0,-0.00000,11.24613
1,1,1,2,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
2,1,1,3,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
3,1,1,4,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
4,1,1,5,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...
85819,2,240,236,0.00000,0.00000,-0.00000,-0.00000,0.0,-0.00000,0.00000
85820,2,240,237,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
85821,2,240,238,0.00000,0.00000,-0.00000,-0.00000,-0.0,-4.16342,0.00000
85822,2,240,239,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000


In [116]:
# Prepare result dataframe, add TD density filename and Kernel filenames

DensList = pd.DataFrame(columns=['beta2(qf)','beta3(qf)', 'beta2(qi)','beta3(qi)', 'Ddens_filename', 'Kernel_filename','Eccen_filename'])

b23_data = pd.read_csv(b23_path,
                    sep=r'\s+',   # 按空格分隔
                    skiprows=0)
for q1 in range(len(b23_data)):
    for q2 in range(len(b23_data)):
        if(Kernels == 2 and q1 != q2): continue
        beta2q1 = b23_data.iloc[q1]['beta2']
        beta3q1 = b23_data.iloc[q1]['beta3']
        beta2q2 = b23_data.iloc[q2]['beta2']
        beta3q2 = b23_data.iloc[q2]['beta3']

        TDdens_filename = f"Proj_{A}{Nuclei}_D1B.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.me"
        if not os.path.exists(os.path.join(output_dir,TDdens_filename)):
            print(TDdens_filename," not exist !")
        
        if(q1 <= q2):
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
            Eccen_filename = f"Proj_{A}{Nuclei}_Eccen.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        else:
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}_{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}.elem"
            Eccen_filename = f"Proj_{A}{Nuclei}_Eccen.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        if not os.path.exists(os.path.join(output_dir,Kernel_filename)):
            print(Kernel_filename," not exist !")
        if not os.path.exists(os.path.join(output_dir,Eccen_filename)):
            print(Eccen_filename," not exist !")
        

        DensList.loc[len(DensList)] = [beta2q1, beta3q1, beta2q2, beta3q2, TDdens_filename, Kernel_filename,Eccen_filename]
DensList

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Ddens_filename,Kernel_filename,Eccen_filename
0,0.0,0.0,0.0,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+000+000_+000+000.me,Proj_20Ne_kern.0D_eMax06.01.01+000+000_+000+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+000+000_+000+0...
1,0.1,0.0,0.1,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+010+000_+010+000.me,Proj_20Ne_kern.0D_eMax06.01.01+010+000_+010+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+010+000_+010+0...
2,0.2,0.0,0.2,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+020+000_+020+000.me,Proj_20Ne_kern.0D_eMax06.01.01+020+000_+020+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+020+000_+020+0...
3,0.3,0.0,0.3,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+030+000_+030+000.me,Proj_20Ne_kern.0D_eMax06.01.01+030+000_+030+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+030+000_+030+0...
4,0.4,0.0,0.4,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+040+000_+040+000.me,Proj_20Ne_kern.0D_eMax06.01.01+040+000_+040+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+040+000_+040+0...
5,0.5,0.0,0.5,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+050+000_+050+000.me,Proj_20Ne_kern.0D_eMax06.01.01+050+000_+050+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+050+000_+050+0...
6,0.6,0.0,0.6,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+060+000_+060+000.me,Proj_20Ne_kern.0D_eMax06.01.01+060+000_+060+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+060+000_+060+0...
7,0.7,0.0,0.7,0.0,Proj_20Ne_D1B.0D_eMax06.01.01+070+000_+070+000.me,Proj_20Ne_kern.0D_eMax06.01.01+070+000_+070+00...,Proj_20Ne_Eccen.0D_eMax06.01.01+070+000_+070+0...


In [117]:
import math
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    Eccen_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Eccen_filename'])
    # read D1B density
    # colspecs = [(0,2),   # Pi (符号)
    #         (2,6),   # J
    #         (6,10),   # K1
    #         (10,14), # K2
    #         (14,18), # ifg
    #         (18,22), # m1
    #         (22,26), # m2
    #         (26,46), # neutron (浮点数，占20列)
    #         (46,64)] # proton
    # Ddens = pd.read_fwf(Ddens_path, colspecs=colspecs, header=0,
    #                 names=['Pi','J','K1','K2','ifg','m1','m2','neutron','proton'])
    Ddens = pd.read_csv(Ddens_path, sep=r'\s+')
    J = 0; Pi = '+'; K1 = 0; K2 = 0
    df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    # calculate r^2
    total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    total_proton = (merged["proton"] * merged["r^2"]).sum()
    Res_data.loc[q1q2,f'r2(total)'] = total_neutron/Norm + total_proton/Norm
    
    # read Eccen kernel
    with open(Eccen_path, 'r') as f:
        line = f.readlines()[1]
        oneBody = float(line[:15].strip())
        twoBody = float(line[15:30].strip())
        # print('Norm=',Norm)
        Res_data.loc[q1q2,f'oneBody'] =  oneBody
        Res_data.loc[q1q2,f'twoBody'] =  twoBody



    # calculate r^4
    total_neutron = (merged["neutron"] * merged["r^4"]).sum()
    total_proton = (merged["proton"] * merged["r^4"]).sum()
    Res_data.loc[q1q2,f'r4(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^2 Y20
    total_neutron = (merged["neutron"] * merged["r^2Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^2Y20"]).sum()
    Res_data.loc[q1q2,f'r2Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y20
    total_neutron = (merged["neutron"] * merged["r^4Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y20"]).sum()
    Res_data.loc[q1q2,f'r4Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y40
    total_neutron = (merged["neutron"] * merged["r^4Y40"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y40"]).sum()
    Res_data.loc[q1q2,f'r4Y40(total)'] = total_neutron/Norm + total_proton/Norm
    
    # calculate r^2_perp = 2/3 r^2 - 4/3*sqrt(pi/5)r^2 Y20
    Res_data['r^2_perp'] = 2.0/3.0 * Res_data['r2(total)'] - 4.0/3.0*math.sqrt(math.pi/5.0)*Res_data['r2Y20(total)'] 
    # calculate r^4_perp = 8/15 r^4 - 32/21*sqrt(pi/5)r^4 Y20 + 16/105*sqrt(pi)*r^4 Y40
    Res_data['r^4_perp'] = 8.0/15.0 * Res_data['r4(total)'] - 32.0/21.0*math.sqrt(math.pi/5.0)*Res_data['r4Y20(total)'] + 16.0/105.0*math.sqrt(math.pi)*Res_data['r4Y40(total)']
     
Res_data.drop(columns=['Ddens_filename', 'Kernel_filename','Eccen_filename'])

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),r2(total),oneBody,twoBody,r4(total),r2Y20(total),r4Y20(total),r4Y40(total),r^2_perp,r^4_perp
0,0.0,0.0,0.0,0.0,154.908414,699.34799,-99.023509,1758.657474,0.000001,0.000550,0.000171,103.272275,937.950035
1,0.1,0.0,0.1,0.0,154.942702,699.98568,-77.884520,1760.266098,5.065904,72.884484,5.495397,97.941045,852.257760
2,0.2,0.0,0.2,0.0,155.144918,703.08628,-14.543060,1768.071111,10.131807,147.715806,25.213219,92.721767,771.359388
3,0.3,0.0,0.3,0.0,156.240539,716.49651,91.773251,1801.839422,15.197723,228.233193,69.113152,88.098079,703.971377
4,0.4,0.0,0.4,0.0,158.527105,743.61954,236.694510,1870.395094,20.263626,316.754507,137.301456,84.268368,652.028915
5,0.5,0.0,0.5,0.0,161.930926,783.92693,417.317970,1972.876966,25.329536,413.990978,220.806813,81.183486,611.790595
6,0.6,0.0,0.6,0.0,167.568947,848.87054,661.250440,2139.301778,30.395448,523.346463,294.453074,79.588068,588.354142
7,0.7,0.0,0.7,0.0,173.830579,921.16687,954.491220,2326.872501,35.461361,638.208071,373.834378,78.408391,571.093790


In [120]:
print(Res_data.drop(columns=['Ddens_filename', 'Kernel_filename','Eccen_filename','beta3(qf)', 'beta2(qi)','Eccen_filename','beta3(qi)','r4(total)','r2Y20(total)','r4Y20(total)','r4Y40(total)']))

   beta2(qf)   r2(total)    oneBody     twoBody    r^2_perp    r^4_perp
0        0.0  154.908414  699.34799  -99.023509  103.272275  937.950035
1        0.1  154.942702  699.98568  -77.884520   97.941045  852.257760
2        0.2  155.144918  703.08628  -14.543060   92.721767  771.359388
3        0.3  156.240539  716.49651   91.773251   88.098079  703.971377
4        0.4  158.527105  743.61954  236.694510   84.268368  652.028915
5        0.5  161.930926  783.92693  417.317970   81.183486  611.790595
6        0.6  167.568947  848.87054  661.250440   79.588068  588.354142
7        0.7  173.830579  921.16687  954.491220   78.408391  571.093790
